## Demo code for "Dynamic 3D Gaze from Afar: Deep Gaze Estimation from Temporal Eye-Head-Body Coordinations"

**Extended with RHFD + LSTM-TWIESN hybrid architecture.**

This notebook demonstrates:
1. Loading the pretrained HBNet + enhanced RHFDGazeModule
2. Extracting RHFD features (Gf: fixation frequency, Gd: gaze density)
3. Gaze estimation with probability-first mapping
4. TWIESN temporal smoothing + exponential moving average

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from models.gazenet import GazeNet, RHFDGazeModule
from models.rhfd_features import RHFDFeatureExtractor
from models.twiesn import TWIESN
from models.smoothing import ExponentialSmoothing
from models.utils import generate_sphere_anchors
from dataloader.gafa import create_gafa_dataset

### Load pretrained model

In [ ]:
### Load model with RHFD + TWIESN + EMA enabled

# The new GazeNet accepts configuration flags:
#   use_rhfd_features=True   → enables Gf/Gd RHFD feature extraction
#   use_probability_head=True → enables probability-first gaze mapping (64 anchors)
#   use_twiesn=True           → enables TWIESN temporal smoothing
#   use_ema=True              → enables exponential moving average

model = GazeNet(
    n_frames=7,
    use_rhfd_features=True,
    use_probability_head=True,
    use_twiesn=True,
    use_ema=True,
    ema_alpha=0.3,
    n_anchors=64,
)

# Load pretrained weights (HBNet weights are loaded, new components initialized randomly)
# You can also load original GAFA weights which will be partially applied:
# model.load_state_dict(torch.load('./models/weights/gazenet_GAFA.pth', map_location='cpu')['state_dict'], strict=False)

# For now, we use random initialization (training needed for best results)
model.cuda()
model.eval()

# Display model configuration
print(f"Model configuration:")
print(f"  RHFD features:    {model.use_rhfd_features}")
print(f"  Probability head: {model.use_probability_head}")
print(f"  TWIESN:           {model.use_twiesn}")
print(f"  EMA smoothing:    {model.use_ema}")
print(f"  Anchors:          {model.n_anchors}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

### Load dataset


Our model takes body image, head position, body velocity as inputs. The following form of input is assumedWe assume following input. Note that our model revieves multiple frames and estimates gaze direction for each frame. 
   - body image: Image capturing whole body of a person. torch.Tensor with the size of (number of frames x 3 (RGB channel) x 256 (image height) x 192 (image width))
   - head mask: Binary image indicating the head position of the person. torch.Tensor with the size of (number of frames x 1 x 256 x 192)
   - body velocity: 2D body velocity of body movement in image plane. torch.Tensor with the size of (number of frames x 2 (dx, dy))

Please see the preprocessing script for details [dataloader/gafa.py](dataloader/gafa.py).

In [58]:
sequence = ['living_room/006']

dataset = create_gafa_dataset(n_frames=7, exp_names=sequence, root_dir='./data/preprocessed/')

In [59]:
batch = dataset[0]

print(batch['image'].shape)
print(batch['head_mask'].shape)
print(batch['body_dv'].shape)

torch.Size([7, 3, 256, 192])
torch.Size([7, 1, 256, 192])
torch.Size([7, 2])


### Inference

In [ ]:
image, head_mask, body_dv = batch['image'], batch['head_mask'], batch['body_dv']

with torch.no_grad():
    image = image.cuda().unsqueeze(0)
    head_mask = head_mask.cuda().unsqueeze(0)
    body_dv = body_dv.cuda().unsqueeze(0)
    gaze_res, head_res, body_res = model(image, head_mask, body_dv, hard_mapping=True)

# Extract all outputs
gaze_direction = gaze_res['direction'][0].cpu()          # Final smoothed direction
gaze_confidence = gaze_res['kappa'][0].cpu()              # Confidence (kappa)

# Additional diagnostic outputs (when use_probability_head=True)
if gaze_res.get('probs') is not None:
    gaze_probs = gaze_res['probs'][0].cpu()               # [T, K] probability distribution
    gaze_entropy = gaze_res['entropy'][0].cpu()           # [T, 1] distribution entropy

# Intermediate outputs (when use_twiesn=True)
if gaze_res.get('direction_init') is not None:
    gaze_init = gaze_res['direction_init'][0].cpu()       # LSTM initial estimate
if gaze_res.get('direction_twiesn') is not None:
    gaze_twiesn = gaze_res['direction_twiesn'][0].cpu()   # TWIESN output (pre-EMA)

print("Gaze direction shape:", gaze_direction.shape)
print("Gaze confidence shape:", gaze_confidence.shape)
if gaze_res.get('probs') is not None:
    print("Gaze probability shape:", gaze_probs.shape)
    print("Max prob per frame:", gaze_probs.max(dim=-1).values.numpy())

### Visualization with intermediate outputs

Plot the estimated gaze direction at each stage:
- Green: Final prediction (after TWIESN + EMA)
- Blue: LSTM initial estimate  
- Orange: TWIESN output (before EMA)

In [ ]:
gaze_dir_2d = gaze_direction[0, 0:2].numpy()
gaze_dir_2d /= np.linalg.norm(gaze_dir_2d)

# Also extract initial estimate for comparison
if gaze_res.get('direction_init') is not None:
    gaze_init_2d = gaze_init[0, 0:2].numpy()
    gaze_init_2d /= np.linalg.norm(gaze_init_2d)
    angular_diff = np.degrees(np.arccos(np.clip(np.dot(gaze_dir_2d, gaze_init_2d), -1, 1)))
    print(f"Angular difference (LSTM init → Final): {angular_diff:.2f}°")

In [62]:
def denormalize(image):
    return image.transpose(1, 2, 0) * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])

In [ ]:
vis_image = denormalize(batch['image'].numpy()[0])
head_center_x = 70
head_center_y = 40
head_center = (int(head_center_x), int(head_center_y))

def draw_gaze_arrow(image, direction_2d, center, color, length=50, label=""):
    d2d = direction_2d[:2].copy()
    d2d /= np.linalg.norm(d2d)
    des = (int(center[0] + d2d[0]*length), int(center[1] + d2d[1]*length))
    image = cv2.arrowedLine(image.copy(), center, des, color, 2, tipLength=0.3)
    return image

# Draw arrows at different stages
vis_final = denormalize(batch['image'].numpy()[0])
vis_final = draw_gaze_arrow(vis_final, gaze_direction[0].numpy(), head_center, (0, 255, 0), label="Final")
vis_final = cv2.putText(vis_final, "Green=Final", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

plt.figure(figsize=(6, 6))
plt.imshow(vis_final)
plt.title("Final Gaze Prediction (TWIESN + EMA)")
plt.axis('off')
plt.show()

# Show comparison if intermediate outputs available
if gaze_res.get('direction_init') is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    titles = ['LSTM Initial', 'TWIESN Smooth', 'Final (EMA)']
    dirs = [gaze_init[0].numpy(), gaze_twiesn[0].numpy(), gaze_direction[0].numpy()]
    
    for ax, title, d2d in zip(axes, titles, dirs):
        vis = denormalize(batch['image'].numpy()[0])
        vis = draw_gaze_arrow(vis, d2d[:2], head_center, (0, 255, 0))
        ax.imshow(vis)
        ax.set_title(title)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()